# Inspect the MP-20 training set
Download the pinned Crys-JEPA Hugging Face training CSV, load it, and print its first five rows.
On SOC, run this inside a compute allocation. CSV order is the embedding export order.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "configs" / "jepa" / "mp.yml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook inside the Crys_JEPA repository.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from reference_baselines.reference_set_comparison.generate_embeddings import (
    HERE, TRAIN_FILE, download_data_file,
)
train_path = download_data_file(TRAIN_FILE, HERE / "artifacts" / "hf")
# Optional local override: train_path = repo_root / "mp_20" / "train.csv"
print("Training file:", train_path.relative_to(repo_root))

In [ ]:
# Preserve literal strings such as the valid formula NaN; retain CSV row order.
train = pd.read_csv(train_path, keep_default_na=False)
print(f"Rows: {len(train):,}; columns: {len(train.columns)}")

In [ ]:
print(train.head())

## Load the exported reference embeddings and metadata

These files were generated on the cluster and copied to `outputs/mp20_train/`.
Loading them does not run JEPA again. `embeddings[i]` must describe the same
crystal as `metadata.iloc[i]`; `embedding_row` and `source_row` are positional
indices, while `Unnamed: 0` is an older source index and is not the array index.

In [ ]:
import json
import hashlib
import numpy as np

export_dir = HERE / "outputs" / "mp20_train"
embeddings = np.load(export_dir / "embeddings.npy", mmap_mode="r", allow_pickle=False)
metadata = pd.read_csv(export_dir / "metadata.csv", keep_default_na=False)
provenance = json.loads((export_dir / "provenance.json").read_text(encoding="utf-8"))

assert embeddings.ndim == 2 and embeddings.shape[1] == 512
assert len(embeddings) == len(metadata) == provenance["rows"]
assert list(embeddings.shape) == provenance["embedding_shape"]
assert np.isfinite(embeddings).all()
np.testing.assert_array_equal(metadata.embedding_row, np.arange(len(metadata)))
np.testing.assert_array_equal(metadata.source_row, np.arange(len(metadata)))
np.testing.assert_array_equal(
    metadata.material_id.astype(str).to_numpy(),
    train.iloc[metadata.source_row.to_numpy()].material_id.astype(str).to_numpy(),
)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for filename, key in [("embeddings.npy", "embeddings_sha256"), ("metadata.csv", "metadata_sha256")]:
    assert file_sha256(export_dir / filename) == provenance[key], f"Hash mismatch: {filename}"
assert file_sha256(train_path) == provenance["train_sha256"], "Loaded train CSV differs from export source"

print(f"Embeddings: {embeddings.shape}, dtype={embeddings.dtype}")
print(f"Metadata: {metadata.shape}")
print("PASS: finite vectors, positional indices, material IDs, and exported/source file hashes")
print("Metadata columns:", metadata.columns.tolist())

In [ ]:
columns = ["embedding_row", "source_row", "material_id", "formula", "num_atoms",
           "formation_energy_per_atom", "e_above_hull"]
print(metadata[columns].head().to_string(index=False))
print("\nFirst three embeddings, first eight dimensions only:")
print(embeddings[:3, :8])

## Page 3: which thermodynamic quantities do we have?

Reference: **Crys-JEPA, page 3, Thermodynamic stability, Eqs. (4)-(7)**,
[local paper](../../../crys_jepa.pdf). `metadata.csv` has 14 columns. Energy
labels were copied from the source dataset; the encoder did not calculate them.

| Paper entity | Meaning | Availability in our export |
|---|---|---|
| $C$ | Crystal being considered | Identified by `material_id`; its CIF is in the loaded source `train`, not metadata |
| $N$ | Atoms in the represented unit cell | `num_atoms`; this is the exporter's primitive/reduced cell count |
| $T_i$ | Element species | `elements` and `chemical_system` |
| $k$ | Number of distinct elements | Derivable from `chemical_system` |
| $\mathbf f=(f_1,\ldots,f_k)$ | Atomic fractions, summing to 1 | Derivable from stoichiometric `formula`; `elements` alone does not give the fractions |
| $E_t$ | Total energy of a unit cell, eV | **Not stored** |
| $E_{t/\mathrm{atom}}=E_t/N$ | Total energy per atom, eV/atom | **Not stored** |
| $\mu_i^{\mathrm{ref}}$ | Energy per atom of elemental reference phase $i$ | **Not stored**; zero elemental formation energy does not mean zero raw elemental energy |
| $E_{f/\mathrm{atom}}$ | Formation energy per atom | **Stored:** `formation_energy_per_atom` |
| $P=(\mathbf f,E_{t/\mathrm{atom}})$ | Composition/total-energy entry | Only composition is available; the total-energy coordinate is missing |
| $\mathcal R$ | Reference phases used to construct a hull | We have the MP-20 training bank; the complete phase bank and settings behind the stored hull labels are not contained in metadata |
| $\mathbf f^j,E^j_{f/\mathrm{atom}}$ | Composition and formation energy of reference phase $j$ | Available/derivable for each of our MP-20 rows; this does not make the bank complete |
| $\lambda_j$ | Atom-weighted phase fractions in the optimal decomposition | **Not stored**; require a phase-diagram/linear-programming calculation |
| $E^{\mathrm{hull}}_{f/\mathrm{atom}}(\mathbf f)$ | Lowest compatible mixture formation energy | **Not independently stored.** Algebraically implied by $E_f-\Delta E$ if the two labels use compatible conventions |
| $E^{\mathrm{hull}}_{t/\mathrm{atom}}(\mathbf f)$ | Same hull energy on a total-energy basis | **Not available** without compatible elemental reference energies |
| $\Delta E$ | Energy above the hull | **Stored:** `e_above_hull` |
| $\epsilon$ | Paper's practical screening threshold | Paper setting: $0.1$ eV/atom; not a metadata column |

`band_gap`, `spacegroup.number`, and the 512-dimensional embedding are also
available, but they are not terms in Eqs. (4)-(7).

### Example: composition is a ratio, not a list of elements

For `Na3MnCoNiO6`, the formula has 12 atoms: Na has fraction $3/12$, O has
fraction $6/12$, and Mn, Co, Ni each have fraction $1/12$. Doubling the unit cell
changes $N$ and the cell total energy, but not these fractions or per-atom energies.

The optional cell below uses pymatgen, which is already a dependency of the
embedding exporter. The energy explanation and array loading do not require it.

In [ ]:
row = metadata.iloc[0]
print("Material:", row.material_id, " Formula:", row.formula)
print("Distinct element count:", len(row.chemical_system.split("-")))
try:
    from pymatgen.core import Composition
except ImportError:
    print("Optional fraction parsing skipped: install pymatgen in this notebook's kernel.")
else:
    fractions = Composition(row.formula).fractional_composition.as_dict()
    assert np.isclose(sum(fractions.values()), 1.0)
    print("Atomic fractions:", fractions)

## Formation energy: comparison with separated elements

DFT (or another energy model) first gives a cell total energy $E_t$. Divide by
its number of atoms: $E_{t/\mathrm{atom}}=E_t/N$. This absolute energy is tied to
the calculation's reference convention; it is not itself an energy-above-hull label.

The paper then subtracts the energy of the **same amounts of the separated
reference elements**:

$$E_{f/\mathrm{atom}}=E_{t/\mathrm{atom}}-\sum_i f_i\mu_i^{\mathrm{ref}}.$$

Here $\mu_i^{\mathrm{ref}}$ (Greek mu) is a fixed elemental reference energy per
atom, not an embedding coordinate or a learned weight. The $f_i$ are atomic
fractions: for AB, $f_A=f_B=1/2$; for AB2, they are $1/3$ and $2/3$.

**Hypothetical AB example, all energies in eV/atom:**

- $E_{t/\mathrm{atom}}(AB)=-5$.
- $\mu_A^{\mathrm{ref}}=-3$ and $\mu_B^{\mathrm{ref}}=-5$.
- Separated-elements baseline: $0.5(-3)+0.5(-5)=-4$.
- Formation energy: $-5-(-4)=-1$.

The compound is 1 eV/atom lower than its separated elements. **Negative formation
energy does not prove that it beats other compounds or mixtures.** A different
combination of phases may still be lower in energy. All compared energies need
compatible DFT settings and energy corrections.

We already have $E_f$ in metadata. We cannot uniquely recover the missing $E_t$
or $\mu_i^{\mathrm{ref}}$ from that number. Multiplying $E_f$ by `num_atoms` gives
a cell formation energy, not the cell total energy $E_t$.

## Energy above hull: comparison with every allowed competing mixture

This is the usual approximately 0 K energy comparison; finite-temperature
stability requires additional free-energy terms.

Fix the target crystal's overall composition. Consider reference phases whose
elements belong to that chemical system or its subsystems. A candidate containing
A and B can compete against A, B, AB, A2B, AB2, etc. A phase containing C cannot
be a physical decomposition product in this closed A-B system.

For every allowed mixture, calculate the atom-weighted average formation energy:

$$E_{\mathrm{mixture}}=\sum_j\lambda_j E^j_{f/\mathrm{atom}}.$$

The constraints in Eq. (5) mean:

- $\lambda_j\geq0$: amounts cannot be negative.
- $\sum_j\lambda_j=1$: weights are fractions of all atoms in the mixture.
- $\sum_j\lambda_j\mathbf f^j=\mathbf f$: conserve every element's overall fraction.

Choose the **lowest-energy composition-balanced mixture**, not the reference
with the smallest individual energy and not a uniform average. That minimum is
$E^{\mathrm{hull}}_{f/\mathrm{atom}}(\mathbf f)$. In a binary diagram, straight
segments between stable phase points represent these mixtures; their lower
boundary is the convex hull. The vertical energy difference at the target
composition is

$$\Delta E=E_{f/\mathrm{atom}}(C)-E^{\mathrm{hull}}_{f/\mathrm{atom}}(\mathbf f).$$

**Continue the hypothetical AB example:** suppose reference phase A2B has
formation energy $-1.4$ eV/atom, and elemental B has formation energy 0.
The balanced reaction $2AB\rightarrow A_2B+B$ preserves two A and two B atoms.
Three of those four atoms are in A2B, so the $\lambda$ weights are $3/4$ for A2B
and $1/4$ for B, not $1/2$ each. This gives

$$E_{\mathrm{mixture}}=(3/4)(-1.4)+(1/4)(0)=-1.05\;\mathrm{eV/atom}.$$

If this is the lowest compatible mixture, the AB crystal with $E_f=-1.00$ is
$0.05$ eV/atom above the hull. It is favorable relative to separated elements,
but decomposition to A2B + B lowers its energy further.

- On the hull: no lower-energy mixture exists within the chosen reference set.
- Above the hull: a lower-energy mixture exists; the gap measures the possible
  energy lowering per atom in this energy-based comparison.
- The paper uses $\Delta E<0.1$ eV/atom as a practical near-stability cutoff.
  Thus this illustrative AB example passes that cutoff despite lying above the
  hull. This does not make it a strict ground-state phase or guarantee synthesis.
- When a target is compared against a fixed external hull that excludes it, a
  negative difference is possible. If it is then added to a compatible complete
  phase diagram, a genuine lower-energy entry becomes part of the updated hull.

If compatible formation energies and compositions for all reference phases are
already available, **raw total energies and elemental reference energies are not
needed to build that formation-energy hull**. Our MP-20 training subset may
omit competing phases, so a hull rebuilt only from these rows need not reproduce
the supplied `e_above_hull` labels.

In [ ]:
# Hypothetical numbers, NOT a calculation on the MP-20 references.
f_AB = np.array([0.5, 0.5])
f_A2B = np.array([2/3, 1/3])
f_B = np.array([0.0, 1.0])
weights = np.array([0.75, 0.25])
mu_ref = np.array([-3.0, -5.0])
E_total_per_atom = -5.0
E_formation_AB = E_total_per_atom - f_AB @ mu_ref
E_mixture = weights @ np.array([-1.4, 0.0])
assert np.allclose(weights[0] * f_A2B + weights[1] * f_B, f_AB)
assert np.isclose(weights.sum(), 1.0)
E_above_hull = E_formation_AB - E_mixture
assert np.isclose(E_above_hull, 0.05)
print(f"AB formation energy: {E_formation_AB:.2f} eV/atom")
print("Mixture composition:", weights[0] * f_A2B + weights[1] * f_B)
print(f"Lowest mixture energy in this example: {E_mixture:.2f} eV/atom")
print(f"AB energy above hull: {E_above_hull:.2f} eV/atom")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), gridspec_kw={"width_ratios": [1.6, 1]})
ax, zoom = axes
ax.plot([0, 1/3, 1], [0, -1.4, 0], color="#2463a6", marker="o", lw=2,
        label="Reference convex hull")
ax.scatter([0.5], [E_formation_AB], color="#ce563c", s=65, zorder=5, label="Candidate AB")
ax.axhline(0, color="0.7", ls="--", lw=1)
ax.annotate("A", (0, 0), xytext=(8, 8), textcoords="offset points")
ax.annotate("A2B", (1/3, -1.4), xytext=(-10, -20), textcoords="offset points")
ax.annotate("B", (1, 0), xytext=(-14, 8), textcoords="offset points")
ax.annotate("AB: -1.00", (0.5, -1), xytext=(0.56, -0.65), arrowprops={"arrowstyle": "->"})
ax.set(xlabel="Atomic fraction of B", ylabel="Formation energy (eV/atom)",
       title="Hypothetical A-B phase diagram", xlim=(-0.05, 1.05), ylim=(-1.62, 0.23))
ax.legend(loc="lower right", frameon=False)
zoom.hlines([-1.05, -1.0], [0.08, 0.08], [0.75, 0.75], colors=["#2463a6", "#ce563c"], lw=2)
zoom.annotate("", xy=(0.82, -1.0), xytext=(0.82, -1.05),
              arrowprops={"arrowstyle": "<->", "color": "#ce563c", "lw": 1.8})
zoom.text(0.1, -0.993, "Candidate AB: -1.00", fontsize=10)
zoom.text(0.1, -1.065, "A2B + B mixture: -1.05", fontsize=10)
zoom.text(0.85, -1.025, "0.05", va="center", fontsize=10)
zoom.set(xlim=(0, 1.1), ylim=(-1.09, -0.96), xticks=[],
         ylabel="Formation energy (eV/atom)", title="Compare at the SAME composition")
for axis in axes:
    axis.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
figure_path = HERE / "outputs" / "thermodynamics_example.png"
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=180, bbox_inches="tight")
plt.show()

## Why Eq. (7) can also use total energy per atom

Every allowed mixture has the same overall composition $\mathbf f$ as the
candidate. Its elemental baseline therefore equals the candidate's baseline:

$$\sum_j\lambda_j\sum_i f_i^j\mu_i^{\mathrm{ref}}
=\sum_i\left(\sum_j\lambda_j f_i^j\right)\mu_i^{\mathrm{ref}}
=\sum_i f_i\mu_i^{\mathrm{ref}}.$$

Subtracting candidate and mixture formation energies cancels this identical
baseline. Hence

$$E_{f/\mathrm{atom}}(C)-\sum_j\lambda_j E^j_{f/\mathrm{atom}}
=E_{t/\mathrm{atom}}(C)-\sum_j\lambda_j E^j_{t/\mathrm{atom}}.$$

That is Eq. (7). It does **not** mean that the elemental reference energies are
zero, or that we can reconstruct the missing raw total energies from metadata.

## What can we calculate from the two stored energy columns?

For bookkeeping, define the **implied** hull energy as $E_f-\Delta E$. This is
an algebraic rearrangement of Eq. (6), valid physically only if the supplied
formation energies and hull labels use consistent energy/reference conventions.
It does not independently recompute a convex hull, recover decomposition phases,
or validate the labels. Metadata/provenance do not provide the complete source
hull and energy-correction history needed for that audit.

In [ ]:
energy_view = metadata[["material_id", "formula", "formation_energy_per_atom", "e_above_hull"]].copy()
for column in ["formation_energy_per_atom", "e_above_hull"]:
    energy_view[column] = pd.to_numeric(energy_view[column], errors="raise")
    assert np.isfinite(energy_view[column]).all()
energy_view["implied_hull_formation_energy_per_atom"] = (
    energy_view.formation_energy_per_atom - energy_view.e_above_hull
)
print(energy_view.head().to_string(index=False))
first = energy_view.iloc[0]
print(f"\n{first.material_id}: E_f={first.formation_energy_per_atom:.6f}, "
      f"Delta E={first.e_above_hull:.6f}, implied hull="
      f"{first.implied_hull_formation_energy_per_atom:.6f} eV/atom")
print("Stored hull-distance range:", energy_view.e_above_hull.min(), "to", energy_view.e_above_hull.max())

## Consequence for the screening experiment

These rows are **known MP-20 training references**, not newly generated VUN
candidates. Their supplied near-stability labels do not measure JEPA screening
performance. A 512-dimensional JEPA embedding is a representation, not a direct
calculation of $E_t$, $E_f$, $\mu_i^{\mathrm{ref}}$, or $\Delta E$.

For later reference-selection comparisons, keep the energy conventions and
**evaluation hull fixed across methods**. Changing an embedding reference pool
changes the screening score; it should not change the physical benchmark used to
judge which selected crystals are better. Cross-system analogues can provide
screening context, but cannot act as composition-balanced decomposition phases
when they introduce elements absent from the target.

Sources: [Crys-JEPA, p. 3, Eqs. (4)-(7)](../../../crys_jepa.pdf) and
[Materials Project phase-diagram methodology](https://docs.materialsproject.org/methodology/materials-methodology/thermodynamic-stability/phase-diagrams-pds).

## Inspect sampled MatterSim-relaxed generated crystals

This section reads the small set of release batches copied from `exp_logs/base/mp_20/eval/` on the SOC cluster. Each numbered directory is one generation/relaxation batch; it is not a successive version of one particular crystal. The rows within its two files must remain aligned.

Each batch contains:

- `relaxed_mattersim.json`: one row per attempted generated crystal, including a pymatgen `Structure` dictionary and MatterSim relaxation fields such as `e_relax` and `exception`. The cells below print the exact columns in this release, rather than assuming a schema.
- `indicators_mattersim_mp_20.pt`: a PyTorch serialization of a dictionary with one Boolean-like vector per JSON row. In the downloaded release batches, it stores `vu` (uniqueness) and `vn` (novelty). The VUN screening mask is derived as `vu AND vn`; the original pipeline writes this as `vu * vn` elementwise. These are release-pipeline indicators, not a new DFT calculation and not a numerical stability score.

In [ ]:
sample_root = HERE / "artifacts" / "release_samples" / "base" / "mp_20" / "eval"
if not sample_root.is_dir():
    raise FileNotFoundError(
        f"Release samples not found at {sample_root}. Copy and extract the cluster archive there first."
    )

batch_dirs = sorted(
    (path for path in sample_root.iterdir() if path.is_dir() and path.name.isdigit()),
    key=lambda path: int(path.name),
)
required = ("relaxed_mattersim.json", "indicators_mattersim_mp_20.pt")
missing = [str(batch / name) for batch in batch_dirs for name in required if not (batch / name).is_file()]
if missing:
    raise FileNotFoundError("Incomplete extracted batch(es):\n" + "\n".join(missing))

print("Sample root:", sample_root)
print("Downloaded evaluation batches:", [int(path.name) for path in batch_dirs])
print("Batch count:", len(batch_dirs))

In [ ]:
# Read every sampled JSON file without changing row order.
relaxed_by_batch = {}
summary_rows = []
for batch_dir in batch_dirs:
    batch = int(batch_dir.name)
    frame = pd.read_json(batch_dir / "relaxed_mattersim.json")
    relaxed_by_batch[batch] = frame
    summary_rows.append({
        "batch": batch,
        "rows": len(frame),
        "columns": ", ".join(frame.columns),
        "exception_rows": int(frame["exception"].astype(bool).sum()) if "exception" in frame else pd.NA,
        "finite_e_relax_rows": int(np.isfinite(pd.to_numeric(frame["e_relax"], errors="coerce")).sum()) if "e_relax" in frame else pd.NA,
    })

json_summary = pd.DataFrame(summary_rows).sort_values("batch")
display(json_summary)

example_batch = int(batch_dirs[0].name)
example_relaxed = relaxed_by_batch[example_batch]
print(f"Exact JSON columns for batch {example_batch}:", example_relaxed.columns.tolist())
print("\nColumn dtypes:")
display(example_relaxed.dtypes.rename("dtype").to_frame())

if "structure" in example_relaxed and len(example_relaxed):
    structure_dict = example_relaxed.iloc[0]["structure"]
    print("Top-level keys in the first pymatgen Structure dictionary:", list(structure_dict))
    print("Sites in the first relaxed structure:", len(structure_dict.get("sites", [])))

# Show scalar metadata but intentionally omit the large nested structure dictionary.
display(example_relaxed.drop(columns=["structure"], errors="ignore").head())

In [ ]:
import torch

indicator_rows = []
indicators_by_batch = {}
for batch_dir in batch_dirs:
    batch = int(batch_dir.name)
    # The official release serializes NumPy arrays, so PyTorch's restricted
    # weights_only loader cannot read it. Use this only for the downloaded,
    # hash-verified Hugging Face release file; never for an untrusted .pt file.
    indicators = torch.load(
        batch_dir / "indicators_mattersim_mp_20.pt", map_location="cpu", weights_only=False
    )
    allowed_keys = {"vu", "vn", "vun"}
    if not {"vu", "vn"}.issubset(indicators) or not set(indicators).issubset(allowed_keys):
        raise ValueError(f"Unexpected indicator keys in batch {batch}: {sorted(indicators)}")

    vu = torch.as_tensor(indicators["vu"]).bool().flatten()
    vn = torch.as_tensor(indicators["vn"]).bool().flatten()
    derived_vun = vu & vn
    # The downloaded release contains only vu and vn. Some regenerated runs
    # may also save vun explicitly, in which case verify it agrees.
    vun = torch.as_tensor(indicators["vun"]).bool().flatten() if "vun" in indicators else derived_vun
    n_rows = len(relaxed_by_batch[batch])
    assert len(vu) == len(vn) == len(vun) == n_rows, f"Row mismatch in batch {batch}"
    assert torch.equal(vun, derived_vun), f"VUN is not vu AND vn in batch {batch}"
    indicators_by_batch[batch] = {"vu": vu, "vn": vn, "vun": vun}
    indicator_rows.append({
        "batch": batch, "rows": n_rows,
        "unique (vu)": int(vu.sum()),
        "novel (vn)": int(vn.sum()),
        "unique and novel (vun)": int(vun.sum()),
    })

indicator_summary = pd.DataFrame(indicator_rows).sort_values("batch")
display(indicator_summary)
print("Totals across downloaded batches:")
print(indicator_summary[["rows", "unique (vu)", "novel (vn)", "unique and novel (vun)"]].sum())

### How to read these two files

The JSON supplies the physical candidate representation after a MatterSim relaxation: its lattice, sites, species, fractional coordinates, and the relaxation's recorded values. `e_relax`, when finite and accompanied by no exception, is MatterSim's energy for that relaxed structure. It is a model energy for that geometry; it is not automatically a formation energy or an energy above the convex hull.

The `.pt` file supplies the two same-row masks made by the release's VUN evaluator. In the original `IV_relax.py` it first builds the relaxed-structure summaries, then computes uniqueness (`vu`) and novelty (`vn`) against the MP-20 reference setup. The downloaded files store those two masks; `VUN = vu AND vn` is derived in `V_screen.py`. It does not store one embedding per row or a further relaxation trajectory.

For subsequent retrieval experiments, retain a row only when its `vun` entry is true, but keep the original batch and within-batch row index alongside it. That preserves the ability to join a selected candidate back to its exact relaxed structure and energy record.

## MEGNet, eSEN and overall screening results: batches 20–29

This replaces the earlier MEGNet-only analysis with **one shared analysis path** for both proxies. The reusable functions live in `analysis.py`; the cells below load and display their results. No weights are loaded and no inference is repeated. All source report hashes are verified before analysis.

Primary endpoint: top **1%**, `unique_refill`, predicted fixed-hull distance **<0.1 eV/atom**. The same selections are judged by both models. Neighbour count **q = 5/10/15/25**, selected fraction **p = 1/5/10%**, and hull threshold are distinct parameters.

**Union clarification:** `union_q` is the deduplicated union of **MP-20 reference crystals** selected by chemical eligibility and kNN. It changes the mean squared embedding-distance screening score. It is **not** a union of generated candidate selections, and does **not** supply the phases for proxy hull evaluation. Every method is evaluated against the same fixed MP2023 phase bank for each proxy. A hull computed from each screening pool would be a different experiment, with method-dependent labels.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

proxy_repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / 'configs/jepa/mp.yml').is_file()), None)
if proxy_repo is None:
    raise FileNotFoundError('Open this notebook inside Crys_JEPA.')
if str(proxy_repo) not in sys.path:
    sys.path.insert(0, str(proxy_repo))
from reference_baselines.reference_set_comparison import analysis as proxy_analysis

proxy_report = proxy_repo / 'reference_baselines/reference_set_comparison/outputs/comparison_20_29/reports'
proxy_bundle = proxy_analysis.load_report(proxy_report)
proxy_figures = proxy_report / 'analysis'
proxy_provenance = proxy_bundle.provenance
print('PASS: every source report file matches its cluster completion receipt.')
print('Models:', proxy_bundle.proxies)
print('Candidate union evaluated:', proxy_provenance['selection']['evaluation_union_count'])
print('Fixed MP2023 reference phases:', proxy_provenance['phases']['phase_count'])
print('Reference structure import failures:', len(proxy_provenance['phases']['structure_failures']))
display(pd.DataFrame([{'proxy': name, 'revision': value['weights']['revision'],
                      'fingerprint': value['fingerprint'], 'torch': value['packages']['torch'],
                      'numpy': value['packages']['numpy']}
                     for name, value in proxy_provenance['models'].items()]))

### Equal budgets and missing labels

Precision is **passing / valid labels**. Confirmed yield is **passing / requested slots**; its upper bound also treats selected unknowns as passing. These denominators must not be mixed. Unknown eSEN labels are neither failures nor successes. They may arise from candidate predictions, required phase predictions, missing elemental references, or hull errors; the downloaded aggregate reports do not identify their exact causes. The cluster's `esen_labels.csv`, `esen_hull_coverage.csv` and prediction status contain that diagnosis.

We audit counts, equal budgets, coverage, overlaps from candidate membership, cross-model consensus, and pooled aggregation. A ten-folder group is the primary ranking unit. With only group 2, pooled and turn tables describe **one comparison**, not independent replications.

In [ ]:
proxy_audit = proxy_analysis.audit_report(proxy_bundle)
display(proxy_audit.round(4))
print('PASS count identities, percentage denominators, equal requested budgets, membership overlaps, consensus and pooled aggregation.')
print('Primary turn groups:', sorted(proxy_bundle.tables['turn_metrics']['group'].unique()))
print('All methods fill their requested budgets; MEGNet has full coverage, eSEN has some unknown labels.')

# Check that the new combined report retains the frozen MEGNet results.
old_megnet_report = proxy_report / 'megnet'
if (old_megnet_report / 'complete.json').is_file():
    old_megnet = proxy_analysis.load_report(old_megnet_report)
    metric_keys = ['scope', 'group', 'method', 'policy', 'fraction', 'proxy', 'threshold_eV_per_atom']
    old_values = old_megnet.tables['metrics'].sort_values(metric_keys).reset_index(drop=True)
    new_values = proxy_bundle.tables['metrics'].query("proxy == 'megnet'").sort_values(metric_keys).reset_index(drop=True)
    pd.testing.assert_frame_equal(old_values, new_values, check_dtype=False)
    assert old_megnet.provenance['models']['megnet']['fingerprint'] == proxy_provenance['models']['megnet']['fingerprint']
    print('PASS: all MEGNet metrics and model identity agree with the earlier standalone report.')

In [ ]:
proxy_primary_columns = ['method', 'stable_count', 'evaluated_count', 'unknown_count', 'budget',
                         'precision_evaluated', 'valid_label_coverage', 'stable_yield_lower_bound',
                         'stable_yield_upper_bound', 'gain_vs_chemical_pp',
                         'paired_chemical_system_ci_low', 'paired_chemical_system_ci_high']
for proxy_name in proxy_bundle.proxies:
    display(Markdown(f'### {proxy_name.upper()}: primary top-1% results'))
    display(proxy_analysis.primary_table(proxy_bundle, proxy_name)[proxy_primary_columns].round(4))
    proxy_analysis.plot_primary(proxy_bundle, proxy_name, proxy_figures)
    plt.show()
display(Markdown('Precision/coverage/yield columns and stored CI endpoints are fractions; `gain_vs_chemical_pp` is in percentage points. '
                 'The plots convert the CI endpoints to percentage points. Intervals resample chemical systems, use valid labels, '
                 'and are descriptive without multiple-comparison adjustment. They compare methods with chemical screening, not with one another.'))

### Budgets, stricter thresholds and duplicates

We report both precision among valid labels and confirmed yield per requested slot. eSEN can rank methods differently under these two criteria when coverage differs. The `<=0` diagnostic differs from the strict `<0.05` and `<0.1` thresholds. Raw and structure-unique selections can also differ; the primary comparison consistently uses unique refill.

In [ ]:
display(proxy_analysis.budget_winners(proxy_bundle).round(4))
for proxy_name in proxy_bundle.proxies:
    values = proxy_analysis.metric_rows(proxy_bundle, proxy_name)
    display(Markdown(f'**{proxy_name.upper()} precision among valid labels across budgets**'))
    display((100*values.pivot(index='method', columns='fraction', values='precision_evaluated')).round(2))
    thresholds = proxy_bundle.tables['pooled_metrics']
    thresholds = thresholds[thresholds.proxy.eq(proxy_name) & thresholds.policy.eq('unique_refill') & np.isclose(thresholds.fraction, .01)]
    display(Markdown('**Top-1% threshold sensitivity** (column 0 means <=0; others mean strictly less than)'))
    display((100*thresholds.pivot(index='method', columns='threshold_eV_per_atom', values='precision_evaluated')).round(2))
    proxy_analysis.plot_budgets(proxy_bundle, proxy_name, proxy_figures)
    plt.show()
raw_unique = proxy_bundle.tables['pooled_metrics']
raw_unique = raw_unique[np.isclose(raw_unique.fraction, .01) & np.isclose(raw_unique.threshold_eV_per_atom, .1)]
print('Raw versus unique-refill primary comparison:')
display(raw_unique[['proxy', 'method', 'policy', 'stable_count', 'evaluated_count', 'unknown_count',
                   'raw_unique_without_refill', 'duplicate_fraction_raw_selection']].round(4))

In [ ]:
proxy_overlap = proxy_bundle.tables['overlap']
proxy_overlap = proxy_overlap[proxy_overlap.scope.eq('pooled') & proxy_overlap.policy.eq('unique_refill') & np.isclose(proxy_overlap.fraction, .01)]
display(proxy_overlap[['method', 'shared', 'baseline_only', 'alternative_only', 'selection_jaccard',
                       'baseline_selected_alternative_not_percent', 'alternative_selected_baseline_not_percent']].round(3))
print('The exclusive percentages use baseline-selected and alternative-selected counts as their respective denominators.')
for proxy_name in proxy_bundle.proxies:
    cohorts = proxy_bundle.tables['paired_comparisons']
    cohorts = cohorts[cohorts.scope.eq('pooled') & cohorts.policy.eq('unique_refill') & cohorts.proxy.eq(proxy_name) & np.isclose(cohorts.fraction, .01)]
    print(proxy_name, 'shared and exclusive cohorts:')
    display(cohorts[['method', 'shared_selected_count', 'shared_stable_count',
                    'baseline_only_selected_count', 'baseline_only_stable_count', 'baseline_only_unknown_count',
                    'alternative_only_selected_count', 'alternative_only_stable_count', 'alternative_only_unknown_count']])
print('At 1%, chemical and kNN-25 share no selected candidates. Selection disagreement is not itself label disagreement.')

In [ ]:
for proxy_name in proxy_bundle.proxies:
    proxy_analysis.plot_batches(proxy_bundle, proxy_name, proxy_figures)
    plt.show()
display(Markdown('Each folder has a separate diagnostic top-1% ranking, with about six requested slots. '
                 'These are not the primary ten-folder selections, and averaging their precision would change the experiment. '
                 'The heatmaps show confirmed yield per requested slot, so missing labels cannot improve the displayed result.'))

In [ ]:
proxy_consensus = proxy_analysis.consensus_table(proxy_bundle)
display(proxy_consensus[['method', 'budget', 'both_evaluated', 'unknown_either', 'both_pass',
                        'megnet_only_pass', 'esen_only_pass', 'neither_pass', 'agreement_fraction',
                        'both_pass_precision_common', 'both_pass_yield_lower', 'both_pass_yield_upper']].round(4))
proxy_analysis.plot_overall(proxy_bundle, proxy_figures)
plt.show()
print('Agreement counts only rows with valid labels from both models; both-pass yield uses every requested slot.')
print('No averaging of MEGNet/eSEN energies, precisions, or ranks is used to declare an overall winner.')

print('Recorded runtime entries (incomplete for killed/timed-out allocations):')
display(proxy_bundle.tables['runtime'].groupby('stage').agg(recorded_seconds=('seconds', 'sum'), entries=('seconds', 'size')).round(2))
proxy_analysis.export_analysis(proxy_bundle, proxy_figures)
print('Saved figures, reusable derived tables and analysis provenance:', proxy_figures)

### What this experiment supports

- **MEGNet:** kNN-25 leads at top 1%, with 56/62 = **90.3%**, versus chemical 41/62 = **66.1%** (+24.2 pp; descriptive 95% interval +8.6 to +40.3 pp). At 5% kNN-5/10 tie; at 10% kNN-10 leads.
- **eSEN:** chemical leads at top 1%, with 37/62 = **59.7%**. kNN-5 gives 31/59 valid = **52.5%**, three unknown; kNN-25 gives 28/59 = **47.5%**, three unknown. Even assigning all missing kNN labels as passing gives 34/62 and 31/62, below chemical's observed 37/62. This missing-label bound does not remove sampling uncertainty: the pure-kNN versus chemical bootstrap intervals all include zero.
- **Budget dependence:** eSEN kNN-5 has the highest observed precision and confirmed slot yield at 5%. At 10%, chemical has the highest valid-label precision, while kNN-10 has the highest confirmed slot yield. These small differences do not establish a universally optimal neighbour count.
- **Both models:** chemical and kNN-5 each have **27/62 = 43.5% confirmed both-pass yield**; kNN-25 has 26/62 = 41.9%. Chemical has no unknowns, while kNN-5/25 each have three. For kNN-25's 59 jointly evaluated rows, 26 pass both, 27 pass only MEGNet, two pass only eSEN, and four fail both: **50.8% label agreement**. The large MEGNet gain is not corroborated by eSEN or a clear gain in confirmed dual-proxy yield.
- **Union:** the deduplicated chemical-plus-kNN **embedding-score pool** does not consistently improve screening under both proxies. The primary union variants have 19–22 confirmed both-pass selections, versus 27 for chemical. They were never used as a separate hull phase bank.

There is **no proxy-independent best method established by this run**. This is one exploratory ten-folder group with 62 primary slots, multiple configurations, and model-dependent labels on MatterSim-relaxed geometries. The proxies share historical materials data; neither is DFT ground truth and their agreement is not proof of synthesizability. Inspect eSEN failure/phase coverage before a larger experiment, freeze the next settings, and test on fresh batches. Selected-union labels cannot establish population recall, full score correlations, or false-negative rates.

All figures and derived tables are stored in `outputs/comparison_20_29/reports/analysis/`. Original downloaded report tables and receipts remain unchanged. The experiment and equations are recorded in `../docs/EXPERIMENT_REPORT.md`.

## UMAP visualizer: all 27,136 MP-20 reference embeddings

The paper's supplied Figure 3 description uses **100,000 crystals sampled from MP and MPtrj**, reduced with UMAP and coloured in five equal-sized formation-energy groups. Here we use **every row of our MP-20 train export** (27,136 x 512), with the same five-group colouring idea. We do not sample down, append energy labels to the input, standardize the embedding dimensions, or apply PCA before UMAP.

The purpose is to inspect whether nearby embedded crystals organise smoothly with formation energy. Exact curves, orientation, separation, and colour boundaries need not match the paper: the populations differ, our lattice scaler was reconstructed, and the supplied figure description does not specify the UMAP hyperparameters. Our choices below are explicit and reproducible.

This section is independent of the cells above. Optional notebook dependencies, if missing, can be installed in the notebook kernel with `%pip install -r requirements-visualization.txt`. Projection coordinates are cached by input hashes, settings, and package versions.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

umap_repo = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / 'configs/jepa/mp.yml').is_file()), None)
if umap_repo is None:
    raise FileNotFoundError('Open this notebook inside Crys_JEPA.')
if str(umap_repo) not in sys.path:
    sys.path.insert(0, str(umap_repo))
from reference_baselines.reference_set_comparison.umap_view import project_reference_export, equal_energy_groups

# Change settings here to explore a separate, independently cached projection.
REF_UMAP_NEIGHBOURS = 15
REF_UMAP_MIN_DIST = 0.1
REF_UMAP_SEED = 2026
umap_export = umap_repo / 'reference_baselines/reference_set_comparison/outputs/mp20_train'
reference_xy, umap_metadata, umap_directory = project_reference_export(
    umap_export, n_neighbors=REF_UMAP_NEIGHBOURS, min_dist=REF_UMAP_MIN_DIST, seed=REF_UMAP_SEED)
print('Projected rows:', len(reference_xy), '| raw dimensions: 512 | UMAP dimensions: 2')
print('Metric: Euclidean; raw unnormalized embeddings; no PCA; one deterministic worker.')
print('Output directory:', umap_directory)

In [ ]:
reference_energy = pd.to_numeric(umap_metadata.formation_energy_per_atom, errors='raise').to_numpy()
energy_groups, energy_group_summary = equal_energy_groups(reference_energy, groups=5)
assert len(reference_xy) == len(energy_groups) == len(umap_metadata)
assert energy_group_summary['count'].sum() == len(umap_metadata)
assert energy_group_summary['count'].max() - energy_group_summary['count'].min() <= 1
print('Five equal-sized groups by formation-energy rank; ties are broken by source-row position.')
print('Ranges may touch if equal energies straddle a rank boundary.')
display(energy_group_summary)
reference_umap_table = umap_metadata.copy()
reference_umap_table['umap_1'] = reference_xy[:, 0]
reference_umap_table['umap_2'] = reference_xy[:, 1]
reference_umap_table['energy_group'] = energy_groups + 1
reference_umap_table.to_csv(umap_directory / 'points.csv', index=False)
energy_group_summary.to_csv(umap_directory / 'energy_groups.csv', index=False)
print('Saved row-aligned coordinates and hover metadata in points.csv.')

In [ ]:
# Colour order follows the screenshot: red, purple, blue, turquoise, yellow.
reference_palette = ['#df876f', '#ad8bda', '#73a4ea', '#7ed5c5', '#d4c15a']
reference_dark = '#091018'
fig, ax = plt.subplots(figsize=(11, 8.5), facecolor=reference_dark)
ax.set_facecolor(reference_dark)
for group, (color, row) in enumerate(zip(reference_palette, energy_group_summary.itertuples())):
    use = energy_groups == group
    ax.scatter(reference_xy[use, 0], reference_xy[use, 1], s=2, color=color,
               alpha=.75, linewidths=0, rasterized=True,
               label=f'G{group+1}: [{row.min_eV_per_atom:.2f}, {row.max_eV_per_atom:.2f}] eV/atom (n={row.count:,})')
ax.set_title(f'Crys-JEPA reference embeddings: {len(reference_xy):,} MP-20 crystals', color='white', fontsize=14, pad=15)
ax.set(xticks=[], yticks=[])
for spine in ax.spines.values():
    spine.set_visible(False)
legend = ax.legend(loc='upper center', bbox_to_anchor=(.5, -.025), ncol=2, fontsize=9,
                   markerscale=5, frameon=True, facecolor='#1a2535', edgecolor='#657186', labelcolor='white')
fig.subplots_adjust(left=.025, right=.975, top=.91, bottom=.19)
fig.text(.5, .025, f'UMAP of raw 512D embeddings | Euclidean | neighbours={REF_UMAP_NEIGHBOURS} | '
         f'min_dist={REF_UMAP_MIN_DIST} | seed={REF_UMAP_SEED}', ha='center', color='#bec7d0', fontsize=9)
fig.savefig(umap_directory / 'umap_energy_quintiles.png', dpi=200, facecolor=fig.get_facecolor(), bbox_inches='tight')
plt.show()

In [ ]:
# Same coordinates, continuous energy colours: inspect smoothness without imposing five categories.
fig, ax = plt.subplots(figsize=(10, 7))
draw_order = np.random.default_rng(REF_UMAP_SEED).permutation(len(reference_xy))
scatter = ax.scatter(reference_xy[draw_order, 0], reference_xy[draw_order, 1],
                     c=reference_energy[draw_order], cmap='turbo', s=2, alpha=.8, linewidths=0, rasterized=True)
fig.colorbar(scatter, ax=ax, label='Source formation energy (eV/atom)')
ax.set(title='Same UMAP projection, coloured by continuous formation energy',
       xlabel='UMAP 1 (arbitrary coordinates)', ylabel='UMAP 2 (arbitrary coordinates)', xticks=[], yticks=[])
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig(umap_directory / 'umap_energy_continuous.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
try:
    import plotly.graph_objects as go
except ImportError:
    print('Optional hover view skipped; install plotly in this notebook kernel to enable it.')
else:
    reference_hover = go.Figure()
    for group, (color, row) in enumerate(zip(reference_palette, energy_group_summary.itertuples())):
        table = reference_umap_table[reference_umap_table.energy_group.eq(group+1)]
        custom = table[['material_id', 'formula', 'formation_energy_per_atom', 'spacegroup.number', 'embedding_row']].to_numpy()
        reference_hover.add_trace(go.Scattergl(x=table.umap_1, y=table.umap_2, mode='markers',
            marker={'size': 3, 'color': color, 'opacity': .8}, customdata=custom,
            name=f'G{group+1}: {row.min_eV_per_atom:.2f} to {row.max_eV_per_atom:.2f} eV/atom',
            hovertemplate='%{customdata[0]} | %{customdata[1]}<br>Formation energy: %{customdata[2]:.4f} eV/atom'
                          '<br>Space group: %{customdata[3]}<br>Embedding row: %{customdata[4]}<extra></extra>'))
    reference_hover.update_layout(template='plotly_dark', title=f'{len(reference_xy):,} MP-20 reference embeddings: zoom and hover',
        paper_bgcolor=reference_dark, plot_bgcolor=reference_dark, height=800, dragmode='pan',
        xaxis={'title': 'UMAP 1', 'showticklabels': False}, yaxis={'title': 'UMAP 2', 'showticklabels': False},
        legend={'orientation': 'h', 'y': -.08})
    html_path = umap_directory / 'umap_interactive.html'
    reference_hover.write_html(html_path, include_plotlyjs=True, full_html=True)
    relative_html = html_path.relative_to(umap_repo / 'reference_baselines/reference_set_comparison').as_posix()
    display(Markdown(f'[Open the standalone interactive UMAP]({relative_html}) - hover for IDs, formula, energy and space group; '
                     'zoom/pan to inspect individual regions. The HTML includes Plotly and works offline.'))
    print('Interactive HTML:', html_path)

### Reading this visual comparison

**Observed in this run:** the projection has curved bands with clear formation-energy organisation. Low-energy red points concentrate toward the upper left; purple and blue regions extend down the right, with turquoise toward the bottom. The central higher-energy yellow region has more colour mixing. This is qualitatively similar to the paper's curved, energy-coloured structure, rather than an exact reproduction of its layout. The continuous-colour view also shows energy transitions without assigning five categories.

Look for neighbouring points with similar energy colours, ordered transitions between energy groups, and whether the continuous colours form smooth bands. Energy labels were added only after fitting, so a colour gradient would be a descriptive sign of energy organisation in the exported embeddings.

Five colours do not imply five learned clusters: the groups were defined by energy rank. UMAP distorts distances and depends on its settings; disconnected islands and large gaps are not proof of physically distinct material families. Do not tune the seed or settings to force the screenshot's shape.

This projection uses MP-20 training references and their source formation-energy labels, not MEGNet/eSEN candidate stability predictions. The paper mixes stable MP structures with structurally unstable trajectory samples, so our smaller reference-only map cannot exactly reproduce its population or energy distribution. Similar-looking bands alone cannot validate the scaler, reproduce the paper quantitatively, or establish stability. Retrieval continues to use the original **512D** embeddings, never these 2D coordinates.